In [3]:
# %% [markdown]
# # Algorithm Performance Comparison: A* vs DHPA*
# This notebook calculates the percentage differences between A* and DHPA* for **Search Phase Time**, **Max Memory**, and **Path Length** across different maps (`maze` and `mp2`).

# %%
import json
import pandas as pd

# Load the datasets
files = {
    'maze': 'maze.json',
    'mp2': 'mp2.json',
    'mp4': 'mp4.json'
}

all_data = []

for map_name, file_path in files.items():
    with open(file_path, 'r') as f:
        content = json.load(f)
        
    for entry in content['data']:
        algo = entry['algorithm']
        timings = entry['timings']
        
        # Mapping metrics based on algorithm structure
        # A* uses 'found_overall_path' for its total/search time.
        # DHPA* uses 'found_overall_path' specifically for its path finding phase.
        search_time = timings.get('found_overall_path', 0)
        max_memory = timings.get('max_memory', 0)
        path_length = timings.get('path_length', 0)
        
        all_data.append({
            'map': map_name,
            'algorithm': algo,
            'search_time': search_time,
            'max_memory': max_memory,
            'path_length': path_length
        })

# Create DataFrame
df = pd.DataFrame(all_data)

# Aggregate metrics by map and algorithm (averaging the 10 runs)
df_avg = df.groupby(['map', 'algorithm']).mean().reset_index()

# Pivot data to easily compare A* and DHPA* side-by-side
pivot_df = df_avg.pivot(index='map', columns='algorithm')
print("--- Aggregated Mean Values ---")
print(pivot_df)
print("\n" + "="*50 + "\n")

# %% [markdown]
# ### Percentage Difference Calculation
# Formula used: 
# $$ \text{Pct Diff} = \frac{\text{DHPA*} - \text{A*}}{\text{A*}} \times 100 $$
# *Negative values indicate DHPA* is smaller/faster/shorter than A*.*

# %%
results = pd.DataFrame(index=pivot_df.index)

metrics = ['search_time', 'max_memory', 'path_length']

for metric in metrics:
    a_val = pivot_df[(metric, 'a')]
    dhpa_val = pivot_df[(metric, 'dhpa')]
    
    # Calculate % difference relative to A*
    results[f'{metric}_pct_diff'] = ((dhpa_val - a_val) / a_val) * 100

# Formatting output for readability
rename_dict = {
    'search_time_pct_diff': 'Search Phase Time Diff (%)',
    'max_memory_pct_diff': 'Max Memory Diff (%)',
    'path_length_pct_diff': 'Path Length Diff (%)'
}
results = results.rename(columns=rename_dict)

print("--- Percentage Difference (DHPA* vs A*) ---")
display(results.round(2))

--- Aggregated Mean Values ---
          search_time           max_memory           path_length        
algorithm           a      dhpa          a      dhpa           a    dhpa
map                                                                     
maze         0.001417  0.000889   4.522266  4.510156      1997.0  2037.0
mp2          0.042408  0.001336  10.747266  3.298437      2871.0  2871.0
mp4          0.086642  0.001781  17.306250  3.737891      4261.0  4271.0


--- Percentage Difference (DHPA* vs A*) ---


,Search Phase Time Diff (%),Max Memory Diff (%),Path Length Diff (%)
map,,,
maze,-37.26,-0.27,2.00
mp2,-96.85,-69.31,0.00
mp4,-97.94,-78.40,0.23
